***[Kaggle Notebook](https://www.kaggle.com/code/ethanyee2706/translate_data)***

# **1&nbsp;&nbsp;&nbsp;Load Dataset**

In [ ]:
import polars as pl

In [ ]:
input_path = "not_translated.csv"

In [ ]:
df = pl.read_csv(input_path)
print(df)

In [ ]:
sentences = df["Sentence"].to_list()

# **2&nbsp;&nbsp;&nbsp;Load Machine Translation Model**

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import torch

In [ ]:
model_name = "facebook/nllb-200-distilled-600M"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# **3&nbsp;&nbsp;&nbsp;Translate Data**

In [ ]:
from tqdm import tqdm

In [ ]:
src_lang = "eng_Latn"
tgt_lang = "vie_Latn"
bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
batch_size = 64
result = []
torch.cuda.empty_cache()

In [ ]:
for i in tqdm(range(0, len(sentences), batch_size), desc="Translating"):
    batch = sentences[i:i+batch_size]
    
    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        src_lang=src_lang
    ).to(model.device)

    with torch.inference_mode():
        tokens = model.generate(
            **inputs,
            forced_bos_token_id=bos_token_id,
            max_new_tokens=128
        )  
        
    outputs = tokenizer.batch_decode(tokens, skip_special_tokens=True)
    result.extend(outputs)

# **4.&ensp;Write to File**

In [ ]:
df = df.with_columns(
    pl.Series(name="translation", values=result)
).write_csv("translated.csv")